In [4]:
from ase.io import read, write
from ase.build import surface

# Load the alpha-quartz CIF
atoms = read('/home/fluffymelon/PANDA/data/SiO2.cif')

# Build a fully periodic unit cell oriented with (1 1 1) // Oxy
# No vacuum; keep periodicity along z as well
oriented = surface(atoms, (1, 1, 1), layers=1, vacuum=0.0, periodic=True)
oriented.set_pbc((True, True, True))
oriented.wrap()

# Save for future reuse
write('/home/fluffymelon/PANDA/data/SiO2_111_oriented.cif', oriented)

# Report
print(oriented)
print('Cell (Ang):', oriented.cell.cellpar())
print('Miller indices oriented:', (1, 1, 1))


Atoms(symbols='Si3O6', pbc=True, cell=[[7.325020452115414, 0.0, 0.0], [2.3782303287716373, 6.928199270172579, 0.0], [0.0, 0.0, 1.8707614791224239]], spacegroup_kinds=..., tags=...)
Cell (Ang): [ 7.32502045  7.32502045  1.87076148 90.         90.         71.05428495]
Miller indices oriented: (1, 1, 1)


In [4]:
from ase.visualize import view

from ase.io import read, write
from ase.build import surface
import numpy as np

# Load the alpha-quartz CIF
atoms = read('/home/fluffymelon/PANDA/examples/building_systems/silica_substrate/SiO2_shift.cif')
Lz = atoms.cell.lengths()[2]
shift = np.array([0.0, 0.0, Lz / 3.0])
atoms.positions += shift

# Interactive 3D visualization in the notebook using ASE's built-in x3d viewer
view(atoms, viewer='x3d', orthographic=True)


In [7]:
from pymatgen.core import Structure
from pymatgen.core.surface import SlabGenerator

# Read CIF with pymatgen
pmg_struct = Structure.from_file('/home/fluffymelon/PANDA/data/SiO2.cif')

# Create (111)-oriented slab with no added vacuum; minimal thickness of one unit repeat
# SlabGenerator uses miller indices in conventional setting
slabgen = SlabGenerator(initial_structure=pmg_struct,
                        miller_index=(1, 1, 1),
                        min_slab_size=1.0,      # thickness in Angstrom; choose ~1 uc
                        min_vacuum_size=0.0,    # no vacuum
                        center_slab=False,
                        in_unit_planes=True)

pmg_slab = slabgen.get_slab()

# Ensure full periodicity flags (pymatgen stores lattice only; periodicity is implicit)
print(pmg_slab)
print('Lattice (abc, alpha beta gamma):', pmg_slab.lattice.abc, pmg_slab.lattice.angles)



Slab Summary (Si3 O6)
Reduced Formula: SiO2
Miller index: (1, 1, 1)
Shift: 0.0000, Scale Factor: [[-1  1  0]
 [-1  0  1]
 [ 0  1  0]]
abc   :   8.512971   7.325020   4.914966
angles:  70.397563  30.000000  54.472858
Sites (9)
1 Si4+     0.468911     1.000000     0.062177
2 Si4+     0.333333     0.666667     0.135578
3 Si4+     0.197755     0.333333     0.802245
4 O2-     0.945886     0.784891     0.467508
5 O2-     0.295170     0.118224     0.560659
6 O2-     0.404271     0.451558     0.326506
7 O2-     0.371497     0.215109     0.897726
8 O2-     0.720781     0.548442     0.423390
9 O2-     0.262395     0.881776     0.324210
Lattice (abc, alpha beta gamma): (8.51297114124272, 7.325020452115414, 4.91496618) (70.3975632021094, 30.000000000000004, 54.47285752710648)


/home/fluffymelon/PANDA/.venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


In [8]:
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import write

# Convert to ASE Atoms and save CIF
ase_atoms = AseAtomsAdaptor.get_atoms(pmg_slab)
ase_atoms.set_pbc((True, True, True))
ase_atoms.wrap()
write('/home/fluffymelon/PANDA/data/SiO2_111_oriented_pmg.cif', ase_atoms)

print(ase_atoms)
print('Cell (Ang):', ase_atoms.cell.cellpar())


MSONAtoms(symbols='Si3O6', pbc=True, cell=[[4.25648557062136, 0.0, 7.372449269999999], [4.2564855706213605, 5.43130114, 2.4574830899999998], [0.0, 0.0, 4.91496618]], bulk_equivalent=..., bulk_wyckoff=..., oxi_states=...)
Cell (Ang): [ 8.51297114  7.32502045  4.91496618 70.3975632  30.         54.47285753]


In [9]:
import os

# Try to fetch alpha-quartz (SiO2) directly from online databases via pymatgen
structure_mp = None
material_id = None

# Preferred: new Materials Project API if available
try:
    from mp_api.client import MPRester as MPClient  # mp-api client
    api_key = os.getenv('MP_API_KEY') or os.getenv('MAPI_KEY')
    with MPClient(api_key=api_key) as mpr:
        # Search for stable SiO2 entries and pick alpha-quartz space groups (152 or 154)
        results = mpr.summary.search(formula='SiO2', fields=['material_id', 'symmetry', 'is_stable'])
        alpha_candidates = [d for d in results if d.is_stable and d.symmetry and d.symmetry.symbol in ('P3_121', 'P3_221')]
        if alpha_candidates:
            material_id = alpha_candidates[0].material_id
            structure_mp = mpr.get_structure_by_material_id(material_id)
except Exception as e:
    structure_mp = None

# Fallback: legacy pymatgen MPRester
if structure_mp is None:
    try:
        from pymatgen.ext.matproj import MPRester
        # api_key = os.getenv('MP_API_KEY') or os.getenv('MAPI_KEY')
        api_key = "WXdCRdJAHy9AEZ23FC5rf6W3b2S30yLk"
        with MPRester(api_key) as mpr:
            # Known alpha-quartz MP id (stable polymorph)
            try:
                material_id = 'mp-7000'
                structure_mp = mpr.get_structure_by_material_id(material_id)
            except Exception:
                # Generic query by formula, then filter by space group
                entries = mpr.query({'formula_pretty': 'SiO2'}, properties=['material_id', 'spacegroup'])
                for ent in entries:
                    if ent.get('spacegroup', {}).get('symbol') in ('P3_121', 'P3_221'):
                        material_id = ent['material_id']
                        structure_mp = mpr.get_structure_by_material_id(material_id)
                        break
    except Exception as e:
        structure_mp = None

if structure_mp is None:
    raise RuntimeError('Failed to fetch alpha-quartz. Please set MP_API_KEY or MAPI_KEY environment variable.')

print('Fetched alpha-quartz from Materials Project:', material_id)
print('Lattice (abc):', structure_mp.lattice.abc)
print('Angles (deg):', structure_mp.lattice.angles)


Fetched alpha-quartz from Materials Project: mp-7000
Lattice (abc): (4.914966, 4.914966353455409, 5.43130114)
Angles (deg): (90.0, 90.0, 119.99999127648242)


In [10]:
from pymatgen.core.surface import SlabGenerator
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import write

# Generate (111)-oriented periodic unit cell (no vacuum) from fetched structure
slabgen = SlabGenerator(initial_structure=structure_mp,
                        miller_index=(1, 1, 1),
                        min_slab_size=1.0,
                        min_vacuum_size=0.0,
                        center_slab=False,
                        in_unit_planes=True)

pmg_slab = slabgen.get_slab()
ase_atoms = AseAtomsAdaptor.get_atoms(pmg_slab)
ase_atoms.set_pbc((True, True, True))
ase_atoms.wrap()

# Save outputs
write('/home/fluffymelon/PANDA/data/SiO2_111_oriented_online.cif', ase_atoms)

print('Saved oriented unit cell to data/SiO2_111_oriented_online.cif')
print(ase_atoms)
print('Cell (Ang):', ase_atoms.cell.cellpar())


Saved oriented unit cell to data/SiO2_111_oriented_online.cif
MSONAtoms(symbols='Si3O6', pbc=True, cell=[[4.256486095, 0.0, -7.3724485286602555], [-2.7445535095356616e-15, 5.431301139999998, -4.9149660000000015], [0.0, 0.0, 4.914966]], bulk_equivalent=..., bulk_wyckoff=..., initial_magmoms=...)
Cell (Ang): [  8.51297076   7.32502033   4.914966   132.14299848 149.99999445
  54.47286064]


In [13]:
# Build oriented slab object for downstream processing (reuse fetched structure if available)
from pymatgen.core.surface import SlabGenerator

base_struct = structure_mp  # from the fetch cell; assumes it ran

slabgen = SlabGenerator(initial_structure=base_struct,
                        miller_index=(1, 1, 1),
                        min_slab_size=6.0,   # few layers to expose surfaces
                        min_vacuum_size=0.0,
                        center_slab=False,
                        in_unit_planes=True)

pmg_slab_proc = slabgen.get_slab()
ase_atoms = AseAtomsAdaptor.get_atoms(pmg_slab_proc)
ase_atoms.set_pbc((True, True, True))
ase_atoms.wrap()

# Save outputs
write('/home/fluffymelon/PANDA/data/SiO2_111_oriented_test.cif', ase_atoms)

print(pmg_slab_proc)
print('Slab lattice (abc, angles):', pmg_slab_proc.lattice.abc, pmg_slab_proc.lattice.angles)


Slab Summary (Si3 O6)
Reduced Formula: SiO2
Miller index: (1, 1, 1)
Shift: 0.0000, Scale Factor: [[-1  1  0]
 [-1  0  1]
 [ 1  0  0]]
abc   :   4.914967   4.914966   5.431301
angles:  90.000000  90.000000 120.000008
Sites (9)
1 O     0.730777     0.144171     0.215109
2 O     0.413394     0.269222     0.881775
3 O     0.855828     0.586605     0.548442
4 O     0.586606     0.855829     0.784891
5 O     0.269223     0.413395     0.451558
6 O     0.144172     0.730778     0.118225
7 Si     0.468911     0.000000     0.000000
8 Si     0.000000     0.468911     0.333333
9 Si     0.531089     0.531089     0.666667
Slab lattice (abc, angles): (4.914966824795097, 4.914966353455409, 5.43130114) (90.0, 90.0, 120.00000793010751)


In [12]:
# Remove undercoordinated surface Si and passivate non-bridging O with H on both faces
import numpy as np
from pymatgen.core import Structure, Element
from pymatgen.analysis.local_env import CrystalNN

slab = pmg_slab_proc.copy()
cnn = CrystalNN(distance_cutoffs=None)

# Helper: get coordination numbers for Si and O
si_indices = [i for i, sp in enumerate(slab.species) if sp.symbol == 'Si']
o_indices = [i for i, sp in enumerate(slab.species) if sp.symbol == 'O']

# Identify Si with <4 O neighbors
si_to_remove = []
for i in si_indices:
    try:
        neigh = cnn.get_nn_info(slab, i)
    except Exception:
        neigh = []
    o_neighbors = [n for n in neigh if n['site'].species_string.startswith('O')]
    if len(o_neighbors) < 4:
        si_to_remove.append(i)

print('Undercoordinated Si to remove:', len(si_to_remove))

# Remove those Si
slab.remove_sites(si_to_remove)

# Recompute indices after removal
o_indices = [i for i, sp in enumerate(slab.species) if sp.symbol == 'O']
si_indices = [i for i, sp in enumerate(slab.species) if sp.symbol == 'Si']

# Identify non-bridging O: O with exactly one Si neighbor
nbo_indices = []
for i in o_indices:
    try:
        neigh = cnn.get_nn_info(slab, i)
    except Exception:
        neigh = []
    si_neighbors = [n for n in neigh if n['site'].species_string.startswith('Si')]
    if len(si_neighbors) == 1:
        nbo_indices.append((i, si_neighbors[0]['site'], si_neighbors[0]['site_index']))

print('Non-bridging O to passivate:', len(nbo_indices))

# Add H for each NBO: place along the outward normal from the bonded Si–O direction
# Parameters
OH_bond = 0.97  # Angstrom

coords_cart = slab.cart_coords
latt = slab.lattice

new_sites = []
for (o_idx, si_site, si_idx) in nbo_indices:
    o_pos = coords_cart[o_idx]
    si_pos = coords_cart[si_idx]
    so_vec = o_pos - si_pos
    norm = np.linalg.norm(so_vec)
    if norm < 1e-6:
        continue
    outward = so_vec / norm
    h_pos = o_pos + outward * OH_bond
    new_sites.append(('H', h_pos))

for sp, pos in new_sites:
    slab.append(sp, latt.get_fractional_coords(pos), coords_are_cartesian=False)

print('Added H atoms:', len(new_sites))

pmg_slab_passivated = slab


Undercoordinated Si to remove: 0
Non-bridging O to passivate: 0
Added H atoms: 0


In [ ]:
# Assign opposite partial charges to top/bottom H and transform to ~15x15x1 nm supercell, then save
import numpy as np
from pymatgen.io.ase import AseAtomsAdaptor
from ase.build import make_supercell
from ase.io import write

# Convert to ASE for charge tagging and supercell convenience
ase_slab = AseAtomsAdaptor.get_atoms(pmg_slab_passivated)
ase_slab.set_pbc((True, True, True))

# Identify H atoms and split into top/bottom by z coordinate
positions = ase_slab.get_positions()
z = positions[:, 2]
z_min, z_max = float(np.min(z)), float(np.max(z))
z_mid = 0.5 * (z_min + z_max)

symbols = ase_slab.get_chemical_symbols()
h_indices = [i for i, s in enumerate(symbols) if s == 'H']

# Example partial charges (tunable): +q on top, -q on bottom, net zero
q = 0.1  # e
charges = np.zeros(len(ase_slab))
for i in h_indices:
    charges[i] = +q if z[i] >= z_mid else -q
ase_slab.set_initial_charges(charges)

# Build supercell ~ 15 x 15 x 1 nm: determine repeats from lattice
cell = ase_slab.get_cell()
a_len = np.linalg.norm(cell[0])
b_len = np.linalg.norm(cell[1])
c_len = np.linalg.norm(cell[2])

# Target nm -> Angstrom
Ax, Ay, Az = 150.0, 150.0, 10.0  # 1 nm = 10 Å; 1 nm thickness target is small; keep ~1 repeat in z
na = max(1, int(np.round(Ax / a_len)))
nb = max(1, int(np.round(Ay / b_len)))
nc = max(1, int(np.round(Az / c_len)))

# Create diagonal supercell
from ase.build import cut
supercell = ase_slab.repeat((na, nb, nc))

print(f'Repeats: na={na}, nb={nb}, nc={nc}')
print('Supercell size (Ang):', supercell.get_cell_lengths_and_angles())

# Save final structures
write('/home/fluffymelon/PANDA/data/SiO2_substrate_final.cif', supercell)
write('/home/fluffymelon/PANDA/data/SiO2_substrate_final.xyz', supercell)

# Optional: also save charges in extended XYZ
from ase.io.extxyz import write_extxyz
with open('/home/fluffymelon/PANDA/data/SiO2_substrate_final_with_charges.xyz', 'w') as f:
    write_extxyz(f, supercell)

print('Saved final substrate files:')
print(' - data/SiO2_substrate_final.cif')
print(' - data/SiO2_substrate_final.xyz')
print(' - data/SiO2_substrate_final_with_charges.xyz')


In [18]:
# ASE-only construction of (111)-oriented alpha-quartz substrate (no vacuum)
from ase.build import surface
from ase.io import read as ase_read, write as ase_write
from pymatgen.io.ase import AseAtomsAdaptor

# Prefer the fetched MP structure if present; else fall back to local CIF
try:
    base_atoms = AseAtomsAdaptor.get_atoms(structure_mp)
except Exception:
    base_atoms = ase_read('/home/fluffymelon/PANDA/data/SiO2.cif')

# Build (111) surface with normal || Oz; keep fully periodic, no vacuum added
# layers controls thickness in units of (hkl) planes; choose a small but >1 thickness
ase_slab = surface(base_atoms, (1, 1, 1), layers=6, vacuum=0.0, periodic=True)
ase_slab.set_pbc((True, True, True))
ase_slab.wrap()

print(ase_slab)
print('Cell (Ang):', ase_slab.get_cell_lengths_and_angles())


MSONAtoms(symbols='Si3O6Si3O6Si3O6Si3O6Si3O6Si3O6', pbc=True, cell=[[7.3250203313384254, 0.0, 0.0], [2.3782308050226306, 6.928199229742565, 0.0], [0.0, 0.0, 13.065563815874714]], bulk_equivalent=..., bulk_wyckoff=..., initial_magmoms=..., tags=...)
Cell (Ang): [ 7.32502033  7.32502057 13.06556382 90.         90.         71.05428132]


/tmp/ipykernel_12093/2373302740.py:19: DeprecationWarning: Please use atoms.cell.cellpar() instead
  print('Cell (Ang):', ase_slab.get_cell_lengths_and_angles())


In [19]:
# Save ASE-oriented (111) unit cell for reuse
ase_write('/home/fluffymelon/PANDA/data/SiO2_111_oriented_ase.cif', ase_slab)
print('Saved to data/SiO2_111_oriented_ase.cif')


Saved to data/SiO2_111_oriented_ase.cif


In [1]:
from ase.io import read, write
from ase.build import surface

# Load the alpha-quartz CIF
atoms = read('/home/fluffymelon/PANDA/data/calcite_unitcell_test.cif')

# Build a fully periodic unit cell oriented with (1 1 1) // Oxy
# No vacuum; keep periodicity along z as well
oriented = surface(atoms, (1, 0, 4), layers=1, vacuum=0.0, periodic=True)
oriented.set_pbc((True, True, True))
oriented.wrap()

# Save for future reuse
write('/home/fluffymelon/PANDA/data/calcite_104_test.cif', oriented)
